In [ ]:
#Donnée désequilibrés 
X_ro,y_ro=RandomOverSampler().fit_resample(X_train, y_train)
print('Classes échantillon oversampled :', dict(pd.Series(y_ro).value_counts()))

In [ ]:
Image ECG
 → split 12 leads
 → U-Net (segmentation)
 → mask → signal pixels
 → pixels → mV
 → resampling à fs
 → submission.parquet


In [1]:
from dataset_ecg import ECGSegDataset


In [2]:
import cv2, torch, numpy as np
from torch.utils.data import Dataset
from utils_ecg import split_12leads

class ECGSegDataset(Dataset):
    def __init__(self, img_paths, mask_paths):
        self.imgs = img_paths
        self.masks = mask_paths

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img = cv2.imread(self.imgs[idx], cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(self.masks[idx], cv2.IMREAD_GRAYSCALE)

        img = cv2.resize(img, (512,512))
        mask = cv2.resize(mask, (512,512))

        img = img / 255.0
        mask = (mask > 127).astype(np.float32)

        img = torch.tensor(img).unsqueeze(0)
        mask = torch.tensor(mask).unsqueeze(0)

        return img.float(), mask.float()


In [3]:
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1):
        super().__init__()
        self.d1 = DoubleConv(in_ch, 64)
        self.d2 = DoubleConv(64, 128)
        self.d3 = DoubleConv(128, 256)
        self.d4 = DoubleConv(256, 512)

        self.pool = nn.MaxPool2d(2)

        self.u3 = DoubleConv(512+256, 256)
        self.u2 = DoubleConv(256+128, 128)
        self.u1 = DoubleConv(128+64, 64)

        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.out = nn.Conv2d(64, out_ch, 1)

    def forward(self, x):
        c1 = self.d1(x)
        c2 = self.d2(self.pool(c1))
        c3 = self.d3(self.pool(c2))
        c4 = self.d4(self.pool(c3))

        x = self.up(c4)
        x = self.u3(torch.cat([x, c3], 1))
        x = self.up(x)
        x = self.u2(torch.cat([x, c2], 1))
        x = self.up(x)
        x = self.u1(torch.cat([x, c1], 1))

        return torch.sigmoid(self.out(x))


In [14]:
class ECGSegDataset(Dataset):
    def __init__(self, img_paths, mask_paths):
        self.imgs = img_paths
        self.masks = mask_paths

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img = cv2.imread(self.imgs[idx], cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(self.masks[idx], cv2.IMREAD_GRAYSCALE)

        if img is None:
            raise FileNotFoundError(self.imgs[idx])
        if mask is None:
            raise FileNotFoundError(self.masks[idx])

        img = cv2.resize(img, (512, 512))
        mask = cv2.resize(mask, (512, 512))

        img = img.astype(np.float32) / 255.0
        mask = (mask > 127).astype(np.float32)

        img = torch.from_numpy(img).unsqueeze(0)
        mask = torch.from_numpy(mask).unsqueeze(0)

        return img, mask

In [4]:
from dataset_ecg import ECGSegDataset
print("ECGSegDataset importée ✔")


ECGSegDataset importée ✔


In [6]:
import torch
from torch.utils.data import DataLoader
from dataset_ecg import ECGSegDataset

from models.unet import UNet
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = ECGSegDataset(train_imgs, train_masks)
loader = DataLoader(dataset, batch_size=4, shuffle=True)

model = UNet().to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(20):
    model.train()
    loss_sum = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        loss = criterion(pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()

    print(f"Epoch {epoch+1} | Loss: {loss_sum/len(loader):.4f}")

torch.save(model.state_dict(), "unet_ecg.pth")


NameError: name 'train_imgs' is not defined

In [7]:
import dataset_ecg
print(dataset_ecg.__file__)


C:\Users\Bureau\Desktop\ecg_pipeline\dataset_ecg.py


In [7]:
unet_ecg_path=r"C:\Users\Bureau\Desktop\ecg_pipeline\data\inference_inputs\7663343\7663343-0001.png"

In [8]:
model.load_state_dict(torch.load("unet_ecg_pth"))


NameError: name 'model' is not defined

In [ ]:
ecg_pipeline/
├── data/
│   ├── inference_inputs
│   ├── train 
│         ├──images
│         │    ├──
│         │
│         └── masks
│   
├── 
│   
├── dataset_ecg.py
├── utils_ecg.py
├── models/
│   └── unet.py
└── train_unet.ipynb